## 1. Notebook Overview

This notebook focuses on cleaning and transforming the raw real estate dataset based on the findings from the data profiling phase.

The goal is to prepare a high-quality dataset suitable for analytics and loading into PostgeSQL.

## 2. Import Libraries

In [5]:
import numpy as np
import pandas as pd

## 3. Load Dataset

In [6]:
df = pd.read_csv("C:/Users/dania/Documents/GIT/real-estate-data-platform/data/raw/realtor-data.csv")

## 4. Data Type Conversion and Data Validation

Review and convert columns to appropiate data types before applying transformations.

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2226382 entries, 0 to 2226381
Data columns (total 12 columns):
 #   Column          Dtype  
---  ------          -----  
 0   brokered_by     float64
 1   status          object 
 2   price           float64
 3   bed             float64
 4   bath            float64
 5   acre_lot        float64
 6   street          float64
 7   city            object 
 8   state           object 
 9   zip_code        float64
 10  house_size      float64
 11  prev_sold_date  object 
dtypes: float64(8), object(4)
memory usage: 203.8+ MB


#### Bathroom Values Inspection

Before converting the bathroom column to a specific data type, we inspect the distinct values to determine whether fractional bathrooms exist in the dataset.

After inspecting, the column bathroom was converted from float64 to Int64.

In [8]:
df['bath'].dropna().sort_values().unique()[:50]

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
       14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26.,
       27., 28., 29., 30., 31., 32., 33., 34., 35., 36., 37., 38., 39.,
       40., 41., 42., 43., 44., 45., 46., 47., 48., 49., 50.])

In [9]:
(df['bath'].dropna() % 1 != 0).sum()

np.int64(0)

In [10]:
#Convert the bathroom column to Int64
df['bath']=df['bath'].astype('Int64')

#### Bedroom Values Inspection

The bedrooms column was validated to confirm that no fractional values exist.
Since all bedroom counts are whole numbers, the column was converted from float64 to Int64.

In [11]:
df['bed'].dropna().sort_values().unique()[:50]

array([ 1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12., 13.,
       14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25., 26.,
       27., 28., 29., 30., 31., 32., 33., 34., 35., 36., 37., 38., 39.,
       40., 41., 42., 43., 44., 45., 46., 47., 48., 49., 50.])

In [12]:
(df['bed'].dropna() % 1).sum()

np.float64(0.0)

In [13]:
df['bed']=df['bed'].astype('Int64')

#### Brokered_by Values Inspection

After validating the brokered_by column and confirming that no fractional value exists, the column was converted from float64 to Int64.

In [14]:
df['brokered_by'].dropna().sort_values().unique()[:50]

array([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10., 11., 12.,
       13., 14., 15., 16., 17., 18., 19., 20., 21., 22., 23., 24., 25.,
       26., 27., 28., 29., 30., 31., 32., 33., 34., 35., 36., 37., 38.,
       39., 40., 41., 42., 43., 44., 45., 46., 47., 48., 49.])

In [15]:
df['brokered_by']=df['brokered_by'].astype('Int64')

#### Previous Sales Date Conversion

The 'prev_sold_date' column was converted from object to datetime64[ns] to enable date-based analysis and time-series operations.

In [16]:
df['prev_sold_date']=pd.to_datetime(df['prev_sold_date'], errors='coerce')

In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2226382 entries, 0 to 2226381
Data columns (total 12 columns):
 #   Column          Dtype         
---  ------          -----         
 0   brokered_by     Int64         
 1   status          object        
 2   price           float64       
 3   bed             Int64         
 4   bath            Int64         
 5   acre_lot        float64       
 6   street          float64       
 7   city            object        
 8   state           object        
 9   zip_code        float64       
 10  house_size      float64       
 11  prev_sold_date  datetime64[ns]
dtypes: Int64(3), datetime64[ns](1), float64(5), object(3)
memory usage: 210.2+ MB


#### ZIP Code Investigation

The ZIP code column contains values with fewer than five digits.

This initially suggested that leading zeros may have been removed during data ingestion. Further investigation revealed that many of these records correspond to properties located in Puerto Rico, where ZIP codes commonly begin with leading zeros.

In addition, one record contained a ZIP code value of `0`, which is not a valid postal code. This value was treated as missing data before standardizing ZIP codes to a five-digit format.

To ensure consistency across all records, the ZIP code column was converted to string format and standardized using zero-padding.

In [18]:
df['zip_code'].head(10)

0    601.0
1    601.0
2    795.0
3    731.0
4    680.0
5    612.0
6    639.0
7    731.0
8    730.0
9    670.0
Name: zip_code, dtype: float64

In [19]:
df['zip_code'].describe()

count    2.226083e+06
mean     5.218668e+04
std      2.895408e+04
min      0.000000e+00
25%      2.961700e+04
50%      4.838200e+04
75%      7.807000e+04
max      9.999900e+04
Name: zip_code, dtype: float64

In [20]:
df['zip_code'].dropna().astype(int).astype(str).str.len().value_counts()

zip_code
5    2102368
4     119719
3       3995
1          1
Name: count, dtype: int64

In [21]:
df[df['zip_code'] < 10000][
    ['city','state','zip_code']
].head(20)

,city,state,zip_code
0,Adjuntas,Puerto Rico,601.0
1,Adjuntas,Puerto Rico,601.0
2,Juana Diaz,Puerto Rico,795.0
3,Ponce,Puerto Rico,731.0
4,Mayaguez,Puerto Rico,680.0
5,San Sebastian,Puerto Rico,612.0
6,Ciales,Puerto Rico,639.0
7,Ponce,Puerto Rico,731.0
8,Ponce,Puerto Rico,730.0
9,Las Marias,Puerto Rico,670.0


In [22]:
df[df['zip_code'] < 100][
    ['city','state','zip_code']
].head(20)

,city,state,zip_code
160666,Balzola,California,0.0


In [23]:
(df['zip_code']==0).sum()

np.int64(1)

In [24]:
df['zip_code'] = df['zip_code'].replace(0,pd.NA)

In [25]:
df['zip_code']=(df['zip_code'].astype('Int64').astype(str).str.zfill(5))

In [26]:
df['zip_code'].head()

0    00601
1    00601
2    00795
3    00731
4    00680
Name: zip_code, dtype: object

#### Street Column Validation

The `street` column was analyzed to determine its data type and uniqueness.

The column is not unique and therefore cannot be used as a primary key. However, all non-null values are whole numbers, indicating that the column likely represents a technical street or address identifier rather than a textual street name.

Since no fractional values were found, the column was converted from `float64` to `Int64`.

In [27]:
df['street'].head(20)

0     1962661.0
1     1902874.0
2     1404990.0
3     1947675.0
4      331151.0
5     1850806.0
6     1298094.0
7     1048466.0
8      734904.0
9     1946226.0
10    1902814.0
11    1773902.0
12    1946165.0
13    1761024.0
14    1879215.0
15      17854.0
16      13687.0
17    1868721.0
18    1312671.0
19       6710.0
Name: street, dtype: float64

In [28]:
df['street'].nunique()

2001358

In [29]:
df['street'].duplicated().sum()

np.int64(225023)

In [30]:
df['street'].is_unique

False

In [31]:
(df['street']%1!=0).sum()

np.int64(10866)

In [32]:
df.loc[
    (df['street'].notna()) &
    (df['street'] % 1 != 0),
    ['street']
].head(20)

,street


In [33]:
df['street'].dropna().head()

0    1962661.0
1    1902874.0
2    1404990.0
3    1947675.0
4     331151.0
Name: street, dtype: float64

In [34]:
df['street'].dropna().head()

0    1962661.0
1    1902874.0
2    1404990.0
3    1947675.0
4     331151.0
Name: street, dtype: float64

In [35]:
df['street']=df['street'].astype('Int64')

In [36]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2226382 entries, 0 to 2226381
Data columns (total 12 columns):
 #   Column          Dtype         
---  ------          -----         
 0   brokered_by     Int64         
 1   status          object        
 2   price           float64       
 3   bed             Int64         
 4   bath            Int64         
 5   acre_lot        float64       
 6   street          Int64         
 7   city            object        
 8   state           object        
 9   zip_code        object        
 10  house_size      float64       
 11  prev_sold_date  datetime64[ns]
dtypes: Int64(4), datetime64[ns](1), float64(3), object(4)
memory usage: 212.3+ MB


#### Price Validation

A small number of properties contain non-positive prices, indicating potential data quality issues.

In [37]:
(df['price']<=0).sum()

np.int64(280)

#### Status Validation

The `status` column was reviewed to identify potential formatting inconsistencies and unexpected category values.

The column contains three valid categories:

- for_sale
- sold
- ready_to_build

No inconsistencies, duplicate categories, or formatting issues were detected.

In [38]:
df['status'].value_counts()

status
for_sale          1389306
sold               812009
ready_to_build      25067
Name: count, dtype: int64

In [39]:
(df['status'].isnull()).sum()

np.int64(0)

In [40]:
df['status']=df['status'].astype('category')

In [41]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2226382 entries, 0 to 2226381
Data columns (total 12 columns):
 #   Column          Dtype         
---  ------          -----         
 0   brokered_by     Int64         
 1   status          category      
 2   price           float64       
 3   bed             Int64         
 4   bath            Int64         
 5   acre_lot        float64       
 6   street          Int64         
 7   city            object        
 8   state           object        
 9   zip_code        object        
 10  house_size      float64       
 11  prev_sold_date  datetime64[ns]
dtypes: Int64(4), category(1), datetime64[ns](1), float64(3), object(3)
memory usage: 197.5+ MB


#### Acre Lot Validation

The relationship between `acre_lot` and `house_size` was investigated to determine whether lot size could be used to estimate missing house size values.

Lot size was converted from acres to square feet and compared with property house size. The resulting correlation was approximately zero, indicating no meaningful linear relationship between the two variables.

Therefore, `acre_lot` was not used to impute missing values in the `house_size` column.

In [42]:
df['acre_lot'].describe()

count    1.900793e+06
mean     1.522303e+01
std      7.628238e+02
min      0.000000e+00
25%      1.500000e-01
50%      2.600000e-01
75%      9.800000e-01
max      1.000000e+05
Name: acre_lot, dtype: float64

In [43]:
df['acre_lot'].quantile([0.5,0.75,0.9,0.95,0.99,0.999])

0.500      0.2600
0.750      0.9800
0.900      5.0000
0.950     14.0200
0.990     98.0000
0.999    896.6976
Name: acre_lot, dtype: float64

In [44]:
df['acre_lot'].isnull().sum()

np.int64(325589)

In [45]:
(df['acre_lot']==0).sum()

np.int64(2226)

In [46]:
(df['house_size'].notna() & df['acre_lot'].notna()).sum()

np.int64(1381897)

In [47]:
df['lot_size_sqft'] = df['acre_lot'] * 43560

In [48]:
df[['house_size', 'lot_size_sqft']].corr()

,house_size,lot_size_sqft
house_size,1.000000,0.000987
lot_size_sqft,0.000987,1.000000


### 5. Column Standardization

Column names were standardized to improve readability and consistency across the project.

In [49]:
df=df.rename(
    columns={
        'bed': 'bedrooms',
        'bath': 'bathrooms'
    }
)

In [50]:
df.head()

,brokered_by,status,price,bedrooms,bathrooms,acre_lot,street,city,state,zip_code,house_size,prev_sold_date,lot_size_sqft
0,103378,for_sale,105000.0,3,2,0.12,1962661,Adjuntas,Puerto Rico,00601,920.0,NaT,5227.2
1,52707,for_sale,80000.0,4,2,0.08,1902874,Adjuntas,Puerto Rico,00601,1527.0,NaT,3484.8
2,103379,for_sale,67000.0,2,1,0.15,1404990,Juana Diaz,Puerto Rico,00795,748.0,NaT,6534.0
3,31239,for_sale,145000.0,4,2,0.10,1947675,Ponce,Puerto Rico,00731,1800.0,NaT,4356.0
4,34632,for_sale,65000.0,6,2,0.05,331151,Mayaguez,Puerto Rico,00680,NaN,NaT,2178.0


### 6. Feature Engineering

#### Price per Square Foot

A new feature called 'price_per_sqft' was created by dividing the property price by the house size.

To ensure data quality, the metric was calculated only for properties with a valid house size greater than zero. Records with missing or invalid house size values were assigned a null value.

In [51]:
(df['house_size']==0).sum()

np.int64(0)

In [52]:
df['house_size'].min()

np.float64(4.0)

In [53]:
df['house_size'].isnull().sum()

np.int64(568484)

In [54]:
df['price_per_sqft'] = np.where(
    (df['house_size'].notna()) & (df['house_size'] > 0),
    df['price'] / df['house_size'],
    np.nan
)

#### Price per Square Foot Validation

The newly created `price_per_sqft` feature was reviewed to identify potential outliers and invalid values.

Several records exhibited unusually high price-per-square-foot values due to either exceptionally high prices or unrealistically small property sizes.

These records will be evaluated during the business rule definition phase before creating the analytical dataset.

In [55]:
df['price_per_sqft'].describe()

count    1.656967e+06
mean     2.803832e+02
std      2.002135e+03
min      0.000000e+00
25%      1.426451e+02
50%      2.013194e+02
75%      3.072917e+02
max      2.426535e+06
Name: price_per_sqft, dtype: float64

In [56]:
df['price_per_sqft'].quantile([0.50,0.75,0.90,0.95,0.99])

0.50     201.319444
0.75     307.291667
0.90     519.428515
0.95     723.101678
0.99    1408.263713
Name: price_per_sqft, dtype: float64

In [57]:
df.nlargest(10, 'price_per_sqft')[['price', 'house_size', 'price_per_sqft', 'city', 'state']]

,price,house_size,price_per_sqft,city,state
221994,2.147484e+09,885.0,2.426535e+06,International,California
1288464,5.150000e+08,1048.0,4.914122e+05,San Diego,California
129837,1.850000e+06,4.0,4.625000e+05,Laurel,New York
72903,8.750000e+08,2440.0,3.586066e+05,Bronx,New York
2064065,9.500000e+07,741.0,1.282051e+05,Beverly Hills,California
1163735,5.000000e+07,1092.0,4.578755e+04,Snowmass,Colorado
1163122,5.500000e+07,1248.0,4.407051e+04,Clifton,Colorado
1307450,1.090000e+08,2514.0,4.335720e+04,Carpinteria,California
1118272,2.954050e+06,100.0,2.954050e+04,Bertram,Texas
1322976,1.000000e+08,3731.0,2.680247e+04,Atherton,California


In [58]:
df[df['price_per_sqft']==0][['price', 'house_size', 'price_per_sqft', 'city', 'state']].head(20)

,price,house_size,price_per_sqft,city,state
46207,0.0,4500.0,0.0,Paterson,New Jersey
46208,0.0,5000.0,0.0,Paterson,New Jersey
286067,0.0,2114.0,0.0,Cary,North Carolina
286068,0.0,1776.0,0.0,Cary,North Carolina
286069,0.0,1888.0,0.0,Cary,North Carolina
286070,0.0,1519.0,0.0,Cary,North Carolina
286072,0.0,2058.0,0.0,Cary,North Carolina
286082,0.0,1812.0,0.0,Cary,North Carolina
286083,0.0,1853.0,0.0,Cary,North Carolina
286084,0.0,1785.0,0.0,Cary,North Carolina


In [59]:
(df['price']<=0).sum()

np.int64(280)

In [60]:
(df['house_size']<100).sum()

np.int64(1)

### 7. Business Rules

Based on the data quality assessment, a set of business rules was defined to create a cleaner analytical dataset.

The following rules will be applied:

- Exclude properties with a price less than or equal to zero.
- Exclude properties with a house size smaller than 100 square feet.
- Exclude properties with a house size greater than 100,000 square feet, as these records were identified as unrealistic outliers during data validation.
- Preserve records with missing house size values.

In [61]:
rows_before = len(df)

rows_after = len(
    df[
        (df['price'] > 0)
        &
        (
            df['house_size'].isna()
            |
            (df['house_size'] >= 100)
        )
    ]
)

print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows after cleaning: {rows_after:,}")
print(f"Rows removed: {rows_before - rows_after:,}")

Rows before cleaning: 2,226,382
Rows after cleaning: 2,224,560
Rows removed: 1,822


### 8. Create Clean Dataset

After validating the data types, standardizing column names, engineering new features, and applying business rules, a clean version of the dataset is created for downstream analysis and database loading.

In [73]:
df_clean = df[
    (df['price'] > 0)
    &
    (
        df['house_size'].isna()
        |
        ((df['house_size'] >= 100) & (df['house_size'] <= 100000))
    )
].copy()

In [74]:
df_clean.describe()

,brokered_by,price,bedrooms,bathrooms,acre_lot,street,house_size,prev_sold_date,lot_size_sqft,price_per_sqft
count,2219931.0,2.224460e+06,1743644.0,1713772.0,1.899600e+06,2213630.0,1.656592e+06,1491535,1.899600e+06,1.656592e+06
mean,52936.838905,5.242480e+05,3.275738,2.496403,1.521543e+01,1012277.498134,2.035493e+03,2017-08-16 16:04:18.894628352,6.627842e+05,2.801672e+02
min,0.0,1.000000e+00,1.0,1.0,0.000000e+00,0.0,1.000000e+02,1901-01-01 00:00:00,0.000000e+00,1.000100e-04
25%,23859.0,1.650000e+05,3.0,2.0,1.500000e-01,506314.0,1.300000e+03,2016-08-09 00:00:00,6.534000e+03,1.426873e+02
50%,52882.0,3.250000e+05,3.0,2.0,2.600000e-01,1012698.5,1.760000e+03,2021-12-01 00:00:00,1.132560e+04,2.013539e+02
75%,79183.0,5.500000e+05,4.0,3.0,9.800000e-01,1521064.75,2.412000e+03,2022-03-04 00:00:00,4.268880e+04,3.073235e+02
max,110142.0,2.147484e+09,473.0,830.0,1.000000e+05,2001357.0,9.999900e+04,2026-04-08 00:00:00,4.356000e+09,2.426535e+06
std,30644.335253,2.139012e+06,1.561271,1.640447,7.630216e+02,583731.362224,1.357695e+03,NaN,3.323722e+07,1.969890e+03


In [75]:
df_clean.shape

(2224460, 14)

In [76]:
df_clean[df_clean['price']<0]

,brokered_by,status,price,bedrooms,bathrooms,acre_lot,street,city,state,zip_code,house_size,prev_sold_date,lot_size_sqft,price_per_sqft


In [77]:
(df_clean['house_size'] < 100).sum()

np.int64(0)

##### Export Clean Dataset

The cleaned dataset is exported as a CSV file to the `data/processed` directory.

This file serves as the standardized input for the next stage of the pipeline, where the data will be loaded into PostgreSQL.

In [78]:
# Export df to csv
df_clean.to_csv("C:/Users/dania/Documents/GIT/real-estate-data-platform/data/processed/usa_real_estate_clean.csv", index=False)

### Cleaning Summary

The dataset was successfully cleaned and prepared for further processing.

The main cleaning steps included:

- Validated data types before conversion.
- Converted identifier columns to nullable integer (`Int64`) data types.
- Converted `prev_sold_date` to `datetime`.
- Standardized ZIP codes to a five-digit format.
- Converted `status` to the `category` data type.
- Renamed `bed` to `bedrooms` and `bath` to `bathrooms`.
- Created the `price_per_sqft` feature.
- Applied business rules to remove properties with non-positive prices and unrealistic house sizes.
- Preserved records with missing house size values to avoid removing potentially valid listings.

The resulting dataset is standardized, consistent, and ready for database loading and analytical processing.